# oai

> Build, inspect, and fork synthetic Codex sessions

In [ ]:
#| default_exp oai

Codex keeps each conversation as a thread: a rollout file of Responses API items under `CODEX_HOME/sessions`, plus an app-server that can create threads, append raw items to their model-visible history, and fork them. Injected items land in the normal rollout, so a synthetic history is indistinguishable from a lived one, and `codex resume <thread-id>` picks it up like any other session.

This module builds the smallest useful Python interface to those operations, and converts authored dialogs into threads through the same pipeline `llmsurgery.ant` uses for Claude: `dlg2canon` recovers canonical messages with real tool calls, and fastllm's Responses denormalizer shapes them as items. Talking to the local `codex` executable happens over app-server's JSONL stdio protocol; creating and editing a session does not itself call a model.

In [ ]:
#| export
import asyncio, json, os, uuid
from pathlib import Path
from fastcore.utils import *
from fastllm.openai_responses import denorm_msgs
from llmsurgery.hist import dlg2canon

## Responses API history

Codex stores the model-visible conversation in the same item format used by the Responses API. Ordinary messages contain `input_text` for a user and `output_text` for an assistant. A tool interaction is a `function_call` followed by a `function_call_output` carrying the same call id.

In [ ]:
#| export
def codex_msg(role, text):
    "A Responses API message item"
    typ = 'input_text' if role=='user' else 'output_text'
    return dict(type='message', role=role, content=[dict(type=typ, text=text)])

def codex_call(name, arguments, call_id=None):
    "A Responses API function call item"
    return dict(type='function_call', name=name, arguments=json.dumps(arguments), call_id=call_id or 'call_'+uuid.uuid4().hex)

def codex_output(call_id, output):
    "The result of a Responses API function call"
    return dict(type='function_call_output', call_id=call_id, output=output)

def tool_turn(prompt, name, arguments, output, answer):
    "A complete synthetic tool-use turn"
    call = codex_call(name, arguments)
    return [codex_msg('user', prompt), call, codex_output(call['call_id'], output), codex_msg('assistant', answer)]

`tool_turn` is enough to plant a worked example: the tool never needs to exist or to have run. The matching call ids are what tie the recorded request and result together.

In [ ]:
sample = tool_turn('Measure the flux.', 'flux_meter', {}, 'flux: 41.7 kilofinches', 'The flux is 41.7 kilofinches.')
sample

## Rollout files

Threads are not purely server-side. Codex records each one under `CODEX_HOME/sessions`, partitioned by date, in a JSONL *rollout*. The thread id is the final component of its filename.

In [ ]:
#| export
CODEX_HOME = Path.home()/'.codex'

def rollout_file(thread_id, codex_home=CODEX_HOME):
    "The persisted rollout for `thread_id`, if it exists"
    return first((Path(codex_home)/'sessions').rglob(f'*{thread_id}.jsonl'))

def load_rollout(thread_id, codex_home=CODEX_HOME):
    "Read the JSONL rollout for `thread_id`"
    return L(dict2obj(json.loads(line)) for line in rollout_file(thread_id, codex_home).read_text().splitlines())

A rollout includes configuration and event records as well as the Responses API items. `load_rollout` deliberately preserves all of them: they are useful when investigating what Codex actually reconstructed on resume.

## Curating a captured rollout

Short examples are easiest to write synthetically. A dojo is long enough that capturing one successful run is safer, after which we can select the useful span, discard reasoning, and make its ids reproducible.

In [ ]:
#| export
def response_items(recs):
    "Extract the model-visible Responses API items from rollout records"
    return L(obj2dict(r)['payload'] for r in recs if r.get('type')=='response_item')

def _item_txts(o):
    if isinstance(o, str): yield o
    elif isinstance(o, dict): yield from (t for v in o.values() for t in _item_txts(v))
    elif is_listy(o): yield from (t for x in o for t in _item_txts(x))

def item_txt(item):
    "Every string in a Responses API item, joined, for finding captured items by text"
    return '\n'.join(_item_txts(obj2dict(item)))

Rollout files contain session metadata, turn contexts, UI events, and model history. Only `response_item` records are sent back as prior conversation. `item_txt` walks nested content and tool outputs, making the boundaries of a useful captured span easy to find.

In [ ]:
#| export
def strip_reasoning(items):
    "Drop Responses API `reasoning` items; replay does not need them"
    return L(items).filter(lambda o: o.get('type')!='reasoning')

def _reid(prefix, key): return prefix+'_'+uuid.uuid5(uuid.NAMESPACE_URL, key).hex

def reid_items(items, key=''):
    "Deterministically re-derive response item and paired call ids, returning fresh dicts"
    items,ids = [obj2dict(o) for o in items],{}
    for i,o in enumerate(items):
        for field in ('id', 'call_id'):
            old = o.get(field)
            if old and old not in ids: ids[old] = _reid(old.partition('_')[0], f'{key}:{i}:{field}')
    def _remap(o):
        if isinstance(o, str): return ids.get(o, o)
        if isinstance(o, dict): return {k:_remap(v) for k,v in o.items()}
        if is_listy(o): return [_remap(v) for v in o]
        return o
    return L(_remap(o) for o in items)

def curate_items(recs, key=''):
    "Extract, strip reasoning from, and re-identify captured rollout records"
    return reid_items(strip_reasoning(response_items(recs)), key)

A reasoning item can contain a large summary and encrypted payload, but is not required for the worked interaction. `reid_items` changes both item ids and call ids, recursively updating outputs that refer to their calls. The same capture and key therefore produce identical injectable history without mutating the capture.

## Talking to app-server

`codex app-server` speaks JSON-RPC over newline-delimited stdin and stdout. A connection starts with one `initialize` request and an `initialized` notification. Responses can be interleaved with event notifications, so the client keeps reading until it sees the id of its own request and retains everything else in `events`.

In [ ]:
#| export
class CodexAppServer:
    "A small async client for `codex app-server`"
    def __init__(self, cmd='codex', codex_home=None):
        self.cmd,self.codex_home = cmd,codex_home
        self.proc,self._id,self.events = None,0,[]

    async def start(self):
        env = os.environ.copy()
        if self.codex_home is not None: env['CODEX_HOME'] = str(self.codex_home)
        self.proc = await asyncio.create_subprocess_exec(self.cmd, 'app-server', stdin=asyncio.subprocess.PIPE,
            stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.DEVNULL, env=env)
        await self.request('initialize', dict(clientInfo=dict(name='fastllm_claude_code', title='fastllm-claude-code', version='0.1')))
        await self.notify('initialized')
        return self

    async def _send(self, message):
        self.proc.stdin.write((json.dumps(message)+'\n').encode())
        await self.proc.stdin.drain()

    async def notify(self, method, params=None): await self._send(dict(method=method, params=params or {}))

    async def request(self, method, params=None):
        self._id += 1
        await self._send(dict(method=method, id=self._id, params=params or {}))
        while line := await self.proc.stdout.readline():
            message = json.loads(line)
            if message.get('id') != self._id:
                self.events.append(message)
                continue
            if 'error' in message: raise RuntimeError(message['error']['message'])
            return dict2obj(message['result'])
        raise RuntimeError('codex app-server exited before replying')

    async def start_thread(self, cwd=None, **kwargs):
        "Start a persisted thread"
        if cwd is not None: kwargs['cwd'] = str(Path(cwd).resolve())
        return (await self.request('thread/start', kwargs)).thread

    async def inject_items(self, thread_id, items):
        "Append Responses API `items` to a loaded thread's history"
        return await self.request('thread/inject_items', dict(threadId=thread_id, items=list(items)))

    async def create_thread(self, items, cwd=None, **kwargs):
        "Start a thread whose history is pre-filled with Responses API `items`"
        thread = await self.start_thread(cwd=cwd, **kwargs)
        await self.inject_items(thread.id, items)
        return thread
    async def fork_thread(self, thread_id, last_turn_id=None, **kwargs):
        "Fork a thread, optionally only through `last_turn_id`"
        params = dict(threadId=thread_id, **kwargs)
        if last_turn_id is not None: params['lastTurnId'] = last_turn_id
        return (await self.request('thread/fork', params)).thread

    async def close(self):
        if self.proc is not None and self.proc.returncode is None:
            self.proc.terminate()
            await self.proc.wait()

    async def __aenter__(self): return await self.start()
    async def __aexit__(self, *args): await self.close()

The high-level operations correspond directly to app-server methods. `start_thread` creates the rollout, `inject_items` appends prebuilt history without starting a model turn, and `fork_thread` copies stored history to a new thread id. `create_thread` composes the first two, which is the usual path from curated items to a ready-to-resume fake dojo session.

In [ ]:
from fastcore.test import *
import tempfile
from fastllm.chat import tool_dtls_tag
from llmsurgery.dialog import *

Before wiring curation to app-server, check it on a miniature captured rollout. Reasoning should disappear, ids should be repeatable, tool calls must remain paired with their outputs, and the source capture must remain untouched.

In [ ]:
captured = [dict(type='response_item', payload=o) for o in [
    sample[0], dict(type='reasoning', id='rs_old', encrypted_content='secret'), *sample[1:]]]
ci = strip_reasoning(response_items(captured))
r1,r2 = reid_items(ci, 'dojo'),reid_items(ci, 'dojo')
test_eq(r1, r2)
test_eq(len(r1), 4)
test_eq(r1[2]['call_id'], r1[1]['call_id'])
test_ne(r1[1]['call_id'], ci[1]['call_id'])
test_eq(ci[1]['call_id'], sample[1]['call_id'])
test_eq([o['type'] for o in ci if 'kilofinches' in item_txt(o)], ['function_call_output', 'message'])

The end-to-end check uses a temporary Codex home. It starts a thread, plants a tool-use example, forks the result, and checks the persisted rollouts. Since no turn is started, this neither contacts a model nor spends tokens.

In [ ]:
#| eval: false
with tempfile.TemporaryDirectory() as d:
    home = Path(d)
    proj = home/'project'; proj.mkdir()  # chkstyle: ignore
    items = tool_turn('Measure the flux.', 'flux_meter', {}, 'flux: 41.7 kilofinches', 'The flux is 41.7 kilofinches.')

    async with CodexAppServer(codex_home=home) as app:
        thread = await app.create_thread(reid_items(items, 'live'), cwd=proj)
        fork = await app.fork_thread(thread.id)

    test_eq(fork.forkedFromId, thread.id)
    for tid in (thread.id, fork.id): assert any('41.7 kilofinches' in repr(r) for r in load_rollout(tid, home))

## From dialogs

The same authored dialog that `llmsurgery.ant.dlg2sess` writes as a Claude session converts to a Codex thread here. `dlg2items` recovers the canonical messages and denormalizes them as Responses items; `dlg2thread` injects them into a fresh persisted thread, ready for `codex resume`. The thread id is minted by codex, so re-converting a dialog makes a new thread rather than overwriting (codex owns its session store; unlike Claude transcripts, we never write rollout files ourselves).

In [ ]:
#| export
def dlg2items(
    dlg, # A `Dialog`, ending with a prompt
    aim_info=None, # Model capability dict for media handling; images enabled if None
):
    "Responses API items for `dlg`, with each reply's tool calls recovered as real items"
    return denorm_msgs(dlg2canon(dlg, aim_info))

async def dlg2thread(
    dlg, # The dialog to convert
    cwd=None, # Project directory for the thread; the current directory if None
    codex_home=None, # Codex home; `~/.codex` if None
    **kwargs, # Passed to `thread/start`
):
    "Create a persisted Codex thread whose history is `dlg`, returning the thread id for `codex resume`"
    async with CodexAppServer(codex_home=codex_home) as app:
        thread = await app.create_thread(reid_items(dlg2items(dlg), dlg.name), cwd=cwd, **kwargs)
    return thread.id

The flux dialog again: a reply that used a tool, plus a referenced image attachment. The tool call comes back as a paired `function_call`/`function_call_output`, and the image as an `input_image`:

In [ ]:
def tool_dtl(func, args, result):
    "A tool-call details block in the reply format `fmt2hist` parses"
    d = json.dumps(dict(id='call1', server=False, call=dict(function=func, arguments=args), result=result))
    return f"{tool_dtls_tag}\n<summary><code>{func}(...)</code></summary>\n\n```json\n{d}\n```\n\n</details>"

png = tiny_png
fdlg = Dialog('flux')
fatt = Attachment(png, 'image/png')
fdlg.mk_message(f'The rig: ![](attachment:{fatt.id})', msg_type=snote, attachments=[fatt])
freply = f"Let me check.\n\n{tool_dtl('flux_meter', {'unit':'kf'}, 'flux: 41.7 kilofinches')}\n\nThe flux is 41.7 kilofinches."
fdlg.mk_message('Measure the flux please.', msg_type=sprompt, output=freply)
fitems = dlg2items(fdlg)
[i['type'] for i in fitems]

In [ ]:
test_eq([i['type'] for i in fitems], ['message', 'message', 'function_call', 'function_call_output', 'message'])
test_eq([i.get('role') for i in fitems][:2], ['user', 'assistant'])
fc = first(i for i in fitems if i['type']=='function_call')
fo = first(i for i in fitems if i['type']=='function_call_output')
test_eq(fo['call_id'], fc['call_id'])
test_eq(fc['name'], 'flux_meter')
assert any(c['type']=='input_image' for c in fitems[0]['content'])

`dlg2thread` needs the `codex` binary (but no key and no network), so like the other app-server cells it stays out of CI. Resuming the thread and asking about the planted fact proves the conversion end to end:

    tid = await dlg2thread(fdlg, proj)
    !codex exec -m gpt-5.6-sol -c model_reasoning_effort=medium resume {tid} "What did the flux_meter tool report, exactly?"

In [ ]:
#| eval: false
tid = await dlg2thread(fdlg, tempfile.mkdtemp())
print(tid, rollout_file(tid))

## Export

In [ ]:
#| hide
#| eval: false
import nbdev; nbdev.nbdev_export()